In [1]:
import glob
import rasterio
from rasterio.merge import merge
import geopandas as gpd
from rasterio.mask import mask

In [3]:
#Gather all individual country binary rasters
file_list = glob.glob(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\coffee_2020\*.tif")
src_files_to_mosaic = [rasterio.open(fp) for fp in file_list]
# 2. Merge rasters spatially into a single regional raster
# Using method='max' ensures that if country borders overlap, presence (1) overrides absence (0)
mosaic, out_transform = merge(
    src_files_to_mosaic, 
    method='max', 
    nodata=0
)
# 3. Update metadata to reflect the expanded regional extent and grid resolution
out_meta = src_files_to_mosaic[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "nodata": 0,
    "compress": "lzw"
})

ValueError: array is too big; `arr.size * arr.dtype.itemsize` is larger than the maximum possible size.

In [ ]:
with rasterio.open(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\BHUTAN\RiceMap\Mosaic_Rice_Bhutan_2022.tif", "w", **out_meta) as dest:
    dest.write(mosaic)
#Close open file connections
for src in src_files_to_mosaic:
    src.close()

In [ ]:
# 1) Read AOI shapefile
aoi = gpd.read_file(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\INDO_AOI.shp")

# 2) Make sure CRS matches the raster
# Use the mosaic file's CRS or the raster you created
with rasterio.open(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\Final Mosaic\Mosaic_Rubber_Indonesia_2024.tif") as src:
    raster_crs = src.crs
    print("Raster CRS:", raster_crs)

aoi = aoi.to_crs(raster_crs)

# 3) Clip the raster to AOI geometry
with rasterio.open(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\Final Mosaic\Mosaic_Rubber_Indonesia_2024.tif") as src:
    shapes = [geom for geom in aoi.geometry]
    clipped, out_transform = mask(
        src,
        shapes=shapes,
        crop=True,
        nodata=0,
        all_touched=False
    )

    out_meta = src.meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": clipped.shape[1],
        "width": clipped.shape[2],
        "transform": out_transform,
        "nodata": 0,
        "compress": "lzw"
    })

# 4) Save clipped output
with rasterio.open(
    r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\INDONESIA\Final Mosaic\Mosaic_Rubber_Indonesia_2024_clipped.tif",
    "w",
    **out_meta
) as dest:
    dest.write(clipped)

In [1]:
import glob
import rasterio
from rasterio.merge import merge
from rasterio.enums import Resampling
from rasterio.transform import Affine

file_list = glob.glob(r"C:\Data Spasial\Rice_GDP\GEE_SEA_Rice_Ci10_20250110\*.tif")

scale_factor = 10 / 100  # 10m -> 100m

downsampled_srcs = []
for fp in file_list:
    src = rasterio.open(fp)
    out_h = max(1, round(src.height * scale_factor))
    out_w = max(1, round(src.width * scale_factor))

    data = src.read(
        out_shape=(src.count, out_h, out_w),
        resampling=Resampling.average  # fraction of 10m presence per 100m cell
    )

    new_transform = src.transform * Affine.scale(
        src.width / out_w, src.height / out_h
    )

    # Wrap as an in-memory dataset so rasterio.merge can use it directly
    memfile = rasterio.io.MemoryFile()
    profile = src.profile.copy()
    profile.update(
        height=out_h, width=out_w, transform=new_transform,
        dtype=data.dtype
    )
    mem_ds = memfile.open(**profile)
    mem_ds.write(data)
    downsampled_srcs.append(mem_ds)
    src.close()

# Now merge — arrays are ~100x smaller, should fit comfortably in memory
mosaic, out_transform = merge(downsampled_srcs, method='max', nodata=0)

out_meta = downsampled_srcs[0].meta.copy()
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "nodata": 0,
    "compress": "lzw"
})

with rasterio.open("SEA_rice_mosaic_100m.tif", "w", **out_meta) as dst:
    dst.write(mosaic)

for ds in downsampled_srcs:
    ds.close()

In [4]:
from pathlib import Path
import rasterio
from rasterio.merge import merge

# Put one or more folders here. Each folder produces one mosaic.
folders = [
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\cocoa_2020"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\coffee_2020"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\palm_2020"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rubber_2020"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\maize_2020"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\rice_2020"),
    Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\commodities_country_processed\2020_data_process\timber_2020"),
]
output_root = Path(r"C:\Data Spasial\FAO_Binary_Commodities_FPDP\mosaicked_all_countries\mosaic_2020_data")
output_root.mkdir(parents=True, exist_ok=True)

created = []
for folder in folders:
    tif_files = sorted(folder.glob("*.tif"))
    if not tif_files:
        print(f"Skipping: no GeoTIFF files found in {folder}")
        continue

    output_path = output_root / f"Mosaic_{folder.name}.tif"
    print(f"Mosaicking {len(tif_files)} files from {folder.name}...")

    with rasterio.open(tif_files[0]) as first_src:
        output_profile = first_src.profile.copy()
        output_dtype = first_src.dtypes[0]

    output_profile.update(
        driver="GTiff",
        dtype=output_dtype,
        nodata=0,
        compress="lzw",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        BIGTIFF="IF_SAFER",
    )

    # dst_path makes rasterio.merge write bounded windows directly to disk.
    src_files = [rasterio.open(path) for path in tif_files]
    try:
        merge(
            src_files,
            method="max",
            nodata=0,
            mem_limit=128,
            dst_path=output_path,
            dst_kwds=output_profile,
        )
    finally:
        for src in src_files:
            src.close()

    created.append(output_path)
    print(f"Created: {output_path}")

print(f"Created {len(created)} mosaic(s)")

Mosaicking 10 files from cocoa_2020...


OverflowError: Python int too large to convert to C long